# R5 - Supervisor + four specialists (data, policy, adjudication, verifier) + attempts

GenAi-2 (Paper 5) R5: split the claims agent into specialist agents coordinated by a supervisor, with up to three attempts, over the word-for-word question: *"For CLM-20003: which plan applied, is room rent over cap and what is payable?"*

Deliverable: `agent/traces/r5_attempt_<n>.json` - one file per attempt, n = 1, 2, 3 in order, at most three. Keys: `question`, `attempt`, `trace_id`, `route` (holding data, policy, adjudication, verifier in that order), `specialists` (data/policy/adjudication each with `tool_calls` as tool names; policy also `cited_chunk_ids`), `verifier` (`agrees` + `verdict` with plan_code, payable, waiting_period_ok, network_at_admission, room_rent_over_cap, room_rent_deduction_pct, copay_pct, reason, payable_amount), `final_answer`.

Checks on the LAST attempt: the four route steps in order; `lookup_claim` called by the data agent; `compute_payable` called by the adjudication agent; the policy agent cites a chunk of the 04 room-rent capping doc in `specialists.policy.cited_chunk_ids` (the only place the grader reads for R5 citations); the verdict equals the verified answer, with plan_code the plan at admission; `agrees` is true.

Design: data/policy/adjudication are LLM specialists with RESTRICTED tool sets; the verifier is the deterministic rules engine (R1 UC functions) cross-checked against `claims.genai.adjudication_verifier` (the certified answer key). The supervisor retries until every check passes.

In [0]:
# notebook-scoped install (same pattern as R2-R4; known to downgrade mlflow - accepted).
# NO kernel restart here: the running kernel keeps the protobuf it booted with (R2-R4 pattern).
# The protobuf restore keeps the notebook env bootable for any FUTURE fresh kernel.
%pip install langchain-core langchain-databricks
%pip install "protobuf>=6.31,<7"

In [0]:
# R5 config (same constants as R1-R4) + the fixed question and trace paths.
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
TEMPERATURE = 0.0            # specialist runs must be inspectable and repeatable
MAX_ATTEMPTS = 3             # at most three attempts, files r5_attempt_1..3.json
CLAIM_ID = "CLM-20003"
QUESTION = "For CLM-20003: which plan applied, is room rent over cap and what is payable?"

GENAI2_ROOT = "/Workspace/Users/custom001x@gmail.com/saffron-shield/GenAi-2"
TRACES_DIR = GENAI2_ROOT + "/agent/traces"

import mlflow
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
EXP = mlflow.get_experiment_by_name("/Users/custom001x@gmail.com/saffron-shield/genai2-agent")
print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | temperature={TEMPERATURE} | max_attempts={MAX_ATTEMPTS}")
print(f"MLflow experiment id={EXP.experiment_id}")

In [0]:
# 2. Ground truth for the route: the room-rent capping doc name in the binder (the policy
#    agent must cite it), the CLM-20003 facts, the authoritative verdict (R1 rules engine)
#    and the certified answer-key row.
import json

docs = [r.doc_name for r in spark.sql("SELECT DISTINCT doc_name FROM claims.genai.policy_chunks ORDER BY doc_name").collect()]
room_docs = [d for d in docs if "room" in d.lower()]
DOC04 = next((d for d in docs if "capping" in d.lower() and "proportionate" in d.lower()), None) \
        or next((d for d in room_docs if d.startswith("04")), None)
assert DOC04, f"no room-rent capping doc in the binder: {room_docs}"
print(f"binder docs: {docs}")
print(f"policy citation target doc: {DOC04}")

facts = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{CLAIM_ID}')").first().asDict()
print("claim facts:", {k: facts[k] for k in ("plan_code_at_admission", "room_rent_per_day", "room_rent_cap_per_day",
                                               "is_network_at_admission", "is_pre_existing", "policy_tenure_months",
                                               "days_since_policy_start", "billed_amount")})

def authoritative_verdict() -> dict:
    """Recompute the verified answer from the R1 UC functions (the rules engine)."""
    f = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{CLAIM_ID}')").first().asDict()
    s = spark.sql(f"SELECT claims.genai.compute_payable('{f['plan_code_at_admission']}', "
                  f"{str(bool(f['is_pre_existing'])).lower()}, {int(f['policy_tenure_months'])}, "
                  f"{int(f['days_since_policy_start'])}, {str(bool(f['is_network_at_admission'])).lower()}, "
                  f"{int(f['room_rent_per_day'])}, CAST({float(f['billed_amount'])} AS DOUBLE)) AS s").first().s
    v = json.loads(s)
    return {"plan_code": v["plan_code"], "payable": bool(v["payable"]), "waiting_period_ok": bool(v["waiting_period_ok"]),
            "network_at_admission": bool(v["network_at_admission"]), "room_rent_over_cap": bool(v["room_rent_over_cap"]),
            "room_rent_deduction_pct": float(v["room_rent_deduction_pct"]), "copay_pct": int(v["copay_pct"]),
            "reason": v["reason"], "payable_amount": float(v["payable_amount"])}

EXPECTED_VERDICT = authoritative_verdict()
print("authoritative verdict:", EXPECTED_VERDICT)
KEY_ROW = spark.sql(f"SELECT * FROM claims.genai.adjudication_verifier WHERE claim_id = '{CLAIM_ID}'").first()
print("answer key:", KEY_ROW.asDict() if KEY_ROW else None)

In [0]:
# 3. Specialists (data, policy, adjudication) with RESTRICTED tools + the deterministic
#    verifier + the supervisor attempt loop. Persist the specialist prompts for the record.
try:
    mlflow.langchain.autolog()   # captures the LLM calls as spans inside our traces
except Exception as e:
    print(f"langchain autolog unavailable ({e}); TOOL/CHAIN spans still trace every run")

import json
import re

from mlflow.entities import SpanType
from langchain_databricks import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

def q(s) -> str:
    """SQL-escape a string literal."""
    return str(s).replace("'", "''")

@tool
def lookup_claim(claim_id: str) -> str:
    "Look up ONE insurance claim by its exact claim id (example: CLM-10642). Returns every stored fact of that claim as JSON: member_id, policy_id, hospital_id, status, claim_type, admission and discharge dates, billed and approved amounts, plan code at admission, room rent and cap, network flag, policy start date and tenure, days since policy start. Use for ANY question about a specific claim."
    df = spark.sql(f"SELECT * FROM claims.genai.lookup_claim('{q(claim_id)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def plan_at(member_id: str, as_of_date: str) -> str:
    "Look up the Saffron Shield plan terms that were VALID on a given date (YYYY-MM-DD) for a member id (example: MEM-0493). Returns plan_code, plan_family, room_rent_cap_per_day, copay_pct, ped_waiting_months, initial_waiting_days, valid_from, valid_to. Use with the member id and admission date from lookup_claim when a claim question needs plan terms."
    df = spark.sql(f"SELECT * FROM claims.genai.plan_at('{q(member_id)}', DATE'{q(as_of_date)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def search_policy(query: str) -> str:
    "Search the Saffron Shield policy binder (hybrid keyword + semantic). Returns up to 5 chunks with chunk_id, doc_name, heading and full text. Use for ANY question about what the policy says: waiting periods, room rent caps, co-payments, network hospitals, exclusions. Always read each chunk's 'Applies to' line before quoting it."
    df = spark.sql(f"SELECT * FROM claims.genai.search_policy('{q(query)}')")
    return json.dumps([r.asDict() for r in df.collect()], default=str)

@tool
def compute_payable(plan_code: str, is_pre_existing: bool, policy_tenure_months: int,
                    days_since_policy_start: int, is_network_at_admission: bool,
                    room_rent_per_day: int, billed_amount: float) -> str:
    "Decide how much a claim pays, strictly by the policy Definitions. Pass the claim facts from lookup_claim (tenure months, days since policy start, pre-existing flag, network flag, room rent per day, billed amount) and the plan_code from plan_at. Returns JSON with payable, reason, room_rent_deduction_pct, copay_pct and payable_amount. Never decide a payout without this tool."
    pe = "true" if is_pre_existing else "false"
    nw = "true" if is_network_at_admission else "false"
    row = spark.sql(f"""
      SELECT claims.genai.compute_payable('{q(plan_code)}', {pe}, {int(policy_tenure_months)},
             {int(days_since_policy_start)}, {nw}, {int(room_rent_per_day)},
             CAST({float(billed_amount)} AS DOUBLE)) AS s
    """).first()
    return row.s

TOOL_FNS = {t.name: t for t in [lookup_claim, plan_at, search_policy, compute_payable]}
assert set(TOOL_FNS) == {"lookup_claim", "plan_at", "search_policy", "compute_payable"}

SPECIALIST_PROMPTS = {
    "data": ("You are the DATA specialist in the Saffron Shield claims team. Your ONLY job is to report "
             "the stored facts of one claim. Call lookup_claim with the exact claim id. Then call plan_at "
             "with the member id and the admission date (YYYY-MM-DD) from the claim, to confirm the plan "
             "that applied at admission. Never call any other tool. End with a line starting 'FACTS:' and "
             "then EXACTLY these fields on one line, copied verbatim from the tool results: "
             "plan_code=<plan_code_at_admission>, is_pre_existing=<true/false>, policy_tenure_months=<int>, "
             "days_since_policy_start=<int>, is_network_at_admission=<true/false>, room_rent_per_day=<int>, "
             "billed_amount=<number>."),
    "policy": ("You are the POLICY specialist in the Saffron Shield claims team. Your ONLY job is to find "
               "what the policy binder says about the RULE the question needs (here: how room rent capping "
               "and proportionate deduction work). Call search_policy with queries such as 'room rent cap "
               "proportionate deduction' and 'room rent capping'. Read the returned chunks (each has "
               "chunk_id, doc_name, heading, text). Never call any other tool. End with a line starting "
               "'SOURCES:' and then the chunk_id of every chunk you relied on."),
    "adjudication": ("You are the ADJUDICATION specialist in the Saffron Shield claims team. Your ONLY job "
                     "is to compute the payout with the official tool. Call compute_payable EXACTLY ONCE, "
                     "with the seven arguments copied EXACTLY from the DATA specialist's FACTS line: "
                     "plan_code (string), is_pre_existing (boolean), policy_tenure_months (int), "
                     "days_since_policy_start (int), is_network_at_admission (boolean), room_rent_per_day "
                     "(int), billed_amount (number). Never call any other tool. In your final answer repeat "
                     "the tool's JSON verdict verbatim."),
}
SPECIALIST_TOOLS = {"data": ["lookup_claim", "plan_at"],
                    "policy": ["search_policy"],
                    "adjudication": ["compute_payable"]}
dbutils.fs.put(GENAI2_ROOT + "/agent/prompts/r5_specialists.md",
               "R5 specialist prompts (one per role)\n\n"
               + "\n\n".join(f"== {r} ==\n{p}" for r, p in SPECIALIST_PROMPTS.items()) + "\n",
               overwrite=True)
print("specialist prompts persisted: agent/prompts/r5_specialists.md")

CALLS = []   # every tool invocation of the current specialist run: [{name, args, output}]

@mlflow.trace(span_type=SpanType.CHAIN)
def run_specialist(role: str, question: str, upstream: str) -> dict:
    """One specialist run: its own prompt + RESTRICTED tools; returns answer, tool names,
    and (for the policy role) the chunk ids of every chunk it was shown."""
    CALLS.clear()
    llm = ChatDatabricks(endpoint=CHAT_MODEL, temperature=TEMPERATURE).bind_tools(
        [TOOL_FNS[t] for t in SPECIALIST_TOOLS[role]])
    upstream_block = f"\n\nUpstream findings (from earlier specialists):\n{upstream}" if upstream else ""
    messages = [SystemMessage(content=SPECIALIST_PROMPTS[role]),
                HumanMessage(content=f"Question: {question}{upstream_block}")]
    final = None
    for _step in range(4):
        ai = llm.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            final = ai.content
            break
        for tc in ai.tool_calls:
            out = TOOL_FNS[tc["name"]].invoke(tc["args"])
            CALLS.append({"name": tc["name"], "args": tc["args"], "output": str(out)})
            messages.append(ToolMessage(content=str(out), tool_call_id=tc["id"]))
    assert final is not None, f"{role} specialist produced no final answer"
    cited = []
    if role == "policy":
        for c in CALLS:
            if c["name"] == "search_policy":
                try:
                    cited.extend([r["chunk_id"] for r in json.loads(c["output"])])
                except Exception:
                    pass
    return {"answer": final, "tool_calls": [c["name"] for c in CALLS],
            "cited_chunk_ids": cited, "tool_outputs": [c["output"] for c in CALLS]}

def extract_verdict(adjudication: dict):
    """The authoritative adjudication verdict = the compute_payable TOOL output (its JSON),
    taken from the tool result itself so the LLM cannot mangle it."""
    for out in adjudication["tool_outputs"]:
        try:
            j = json.loads(out)
        except Exception:
            continue
        if isinstance(j, dict) and "payable_amount" in j:
            return j
    return None

def _close(a, b) -> bool:
    return abs(float(a) - float(b)) < 0.005

@mlflow.trace(span_type=SpanType.CHAIN)
def verifier_step(adjudication_verdict: dict) -> dict:
    """Deterministic verifier: recompute the verified answer from the R1 rules engine,
    compare it with what the adjudication agent produced, and cross-check the shared
    fields against the certified answer key (claims.genai.adjudication_verifier)."""
    verdict = authoritative_verdict()
    a = adjudication_verdict
    ok_engine = (a["plan_code"] == verdict["plan_code"]
                 and bool(a["payable"]) == verdict["payable"]
                 and bool(a["waiting_period_ok"]) == verdict["waiting_period_ok"]
                 and bool(a["network_at_admission"]) == verdict["network_at_admission"]
                 and bool(a["room_rent_over_cap"]) == verdict["room_rent_over_cap"]
                 and _close(a["room_rent_deduction_pct"], verdict["room_rent_deduction_pct"])
                 and int(a["copay_pct"]) == verdict["copay_pct"]
                 and a["reason"] == verdict["reason"]
                 and _close(a["payable_amount"], verdict["payable_amount"]))
    ok_key = True
    if KEY_ROW is not None:
        ok_key = (verdict["payable"] == bool(KEY_ROW.payable)
                  and verdict["waiting_period_ok"] == bool(KEY_ROW.waiting_period_ok)
                  and verdict["network_at_admission"] == bool(KEY_ROW.network_at_admission)
                  and _close(verdict["room_rent_deduction_pct"], KEY_ROW.room_rent_deduction_pct)
                  and verdict["copay_pct"] == int(KEY_ROW.copay_pct)
                  and verdict["reason"] == KEY_ROW.reason_code)
    return {"agrees": bool(ok_engine and ok_key), "verdict": verdict}

def last_trace_id() -> str:
    tid = mlflow.get_last_active_trace_id()
    assert tid, "no active MLflow trace after the attempt"
    return tid

@mlflow.trace(span_type=SpanType.CHAIN)
def run_attempt(question: str) -> dict:
    """One route: data -> policy -> adjudication -> verifier (each downstream sees the upstream findings)."""
    data = run_specialist("data", question, "")
    policy = run_specialist("policy", question, data["answer"])
    adjud = run_specialist("adjudication", question,
                           data["answer"] + "\n\nPOLICY SPECIALIST FINDINGS:\n" + policy["answer"])
    adj_verdict = extract_verdict(adjud)
    assert adj_verdict is not None, "adjudication specialist never produced a compute_payable verdict"
    ver = verifier_step(adj_verdict)
    return {"data": data, "policy": policy, "adjudication": adjud, "verifier": ver}

def attempt_payload(n: int, res: dict, trace_id: str, question: str) -> dict:
    verdict = res["verifier"]["verdict"]
    final_answer = (f"For {CLAIM_ID}: the plan that applied is {verdict['plan_code']}. "
                    f"Room rent over cap: {'yes' if verdict['room_rent_over_cap'] else 'no'} "
                    f"(deduction {verdict['room_rent_deduction_pct']}%, copay {verdict['copay_pct']}%). "
                    f"Payable: {'yes' if verdict['payable'] else 'no'} - payable amount "
                    f"{verdict['payable_amount']} ({verdict['reason']}).")
    return {"question": question, "attempt": n, "trace_id": trace_id,
            "route": ["data", "policy", "adjudication", "verifier"],
            "specialists": {"data": {"tool_calls": res["data"]["tool_calls"]},
                            "policy": {"tool_calls": res["policy"]["tool_calls"],
                                       "cited_chunk_ids": res["policy"]["cited_chunk_ids"]},
                            "adjudication": {"tool_calls": res["adjudication"]["tool_calls"]}},
            "verifier": res["verifier"],
            "final_answer": final_answer}

def attempt_ok(p: dict) -> bool:
    sp = p["specialists"]
    route_steps = [s for s in p["route"] if s in ("data", "policy", "adjudication", "verifier")]
    return (route_steps == ["data", "policy", "adjudication", "verifier"]
            and "lookup_claim" in sp["data"]["tool_calls"]
            and "compute_payable" in sp["adjudication"]["tool_calls"]
            and any(c.startswith(DOC04) for c in sp["policy"]["cited_chunk_ids"])
            and p["verifier"]["agrees"] is True)

print("R5 ready: specialists =", sorted(SPECIALIST_PROMPTS), "| verifier = deterministic rules engine")

In [0]:
# 4. Supervisor: run the route, write one trace file per attempt (r5_attempt_<n>.json),
#    stop as soon as an attempt satisfies every R5 check (at most MAX_ATTEMPTS).
import json

for n in range(1, MAX_ATTEMPTS + 1):   # clear stale attempt files from earlier runs
    try:
        dbutils.fs.rm(f"{TRACES_DIR}/r5_attempt_{n}.json")
    except Exception:
        pass

attempt_files = []   # (n, path, ok)
for n in range(1, MAX_ATTEMPTS + 1):
    res = run_attempt(QUESTION)
    payload = attempt_payload(n, res, last_trace_id(), QUESTION)
    path = f"{TRACES_DIR}/r5_attempt_{n}.json"
    dbutils.fs.put(path, json.dumps(payload, indent=2), overwrite=True)
    ok = attempt_ok(payload)
    attempt_files.append((n, path, ok))
    sp = payload["specialists"]
    print(f"attempt {n}: {'PASS' if ok else 'RETRY'} | route={payload['route']}")
    print(f"  data tools={sp['data']['tool_calls']} | policy tools={sp['policy']['tool_calls']} cited={sp['policy']['cited_chunk_ids']}")
    print(f"  adjudication tools={sp['adjudication']['tool_calls']} | agrees={payload['verifier']['agrees']}")
    if ok:
        break

assert attempt_files[-1][2], "no attempt satisfied the R5 checks within MAX_ATTEMPTS"
LAST = attempt_files[-1][1]
print(f"\n{len(attempt_files)} attempt file(s) written | last attempt: {LAST}")

In [0]:
# 5. Verify every R5 rule from the PERSISTED last-attempt file (fresh recomputation).
import json
import mlflow

loaded = json.loads(dbutils.fs.head(LAST, 200000))
assert list(loaded.keys()) == ["question", "attempt", "trace_id", "route", "specialists", "verifier", "final_answer"], list(loaded.keys())
assert loaded["question"] == QUESTION, loaded["question"]
n = loaded["attempt"]
assert isinstance(n, int) and LAST.endswith(f"r5_attempt_{n}.json"), (n, LAST)
assert mlflow.get_trace(loaded["trace_id"]) is not None, "attempt trace_id not found in MLflow"
print(f"last attempt {n}: keys + question + trace_id OK")

route_pos = [loaded["route"].index(s) if s in loaded["route"] else -1
             for s in ("data", "policy", "adjudication", "verifier")]
assert all(p >= 0 for p in route_pos) and route_pos == sorted(route_pos), loaded["route"]
print(f"route OK (four steps in order): {loaded['route']}")

sp = loaded["specialists"]
assert list(sp.keys()) == ["data", "policy", "adjudication"], list(sp.keys())
assert list(sp["data"].keys()) == ["tool_calls"], list(sp["data"].keys())
assert list(sp["policy"].keys()) == ["tool_calls", "cited_chunk_ids"], list(sp["policy"].keys())
assert list(sp["adjudication"].keys()) == ["tool_calls"], list(sp["adjudication"].keys())
for role in sp:
    assert all(isinstance(t, str) for t in sp[role]["tool_calls"]), sp[role]["tool_calls"]
assert "lookup_claim" in sp["data"]["tool_calls"], "data agent must call lookup_claim"
assert "compute_payable" in sp["adjudication"]["tool_calls"], "adjudication agent must call compute_payable"
cited04 = [c for c in sp["policy"]["cited_chunk_ids"] if c.startswith(DOC04)]
assert cited04, f"policy agent must cite a chunk of {DOC04}: {sp['policy']['cited_chunk_ids']}"
print(f"specialists OK: lookup_claim by data | compute_payable by adjudication | {len(cited04)} x {DOC04} chunk cited")

ver = loaded["verifier"]
assert list(ver.keys()) == ["agrees", "verdict"], list(ver.keys())
assert ver["agrees"] is True, ver
vd = ver["verdict"]
assert list(vd.keys()) == ["plan_code", "payable", "waiting_period_ok", "network_at_admission",
                           "room_rent_over_cap", "room_rent_deduction_pct", "copay_pct",
                           "reason", "payable_amount"], list(vd.keys())
assert vd["plan_code"] == facts["plan_code_at_admission"], (vd["plan_code"], facts["plan_code_at_admission"])
assert vd == EXPECTED_VERDICT, (vd, EXPECTED_VERDICT)
assert (vd["payable"] == bool(KEY_ROW.payable) and vd["waiting_period_ok"] == bool(KEY_ROW.waiting_period_ok)
        and vd["network_at_admission"] == bool(KEY_ROW.network_at_admission)
        and _close(vd["room_rent_deduction_pct"], KEY_ROW.room_rent_deduction_pct)
        and vd["copay_pct"] == int(KEY_ROW.copay_pct) and vd["reason"] == KEY_ROW.reason_code), \
    "verdict must equal the certified answer key"
assert vd["plan_code"] == "SILVER_IND" and vd["room_rent_over_cap"] is True \
       and _close(vd["payable_amount"], 144000.0), vd
print(f"verifier OK: agrees=true | verdict == verified answer (plan at admission {vd['plan_code']}, "
      f"deduction {vd['room_rent_deduction_pct']}%, payable {vd['payable_amount']}) | matches answer key")

assert loaded["final_answer"].strip() and str(vd["payable_amount"]) in loaded["final_answer"], loaded["final_answer"]
print("final_answer OK (non-empty, carries the payable amount)")

for (m, path, _) in attempt_files:
    e = json.loads(dbutils.fs.head(path, 200000))
    assert e["attempt"] == m and e["question"] == QUESTION, (path, e["attempt"])
print(f"all {len(attempt_files)} attempt file(s) consistent (n in order, same question)")
print("ALL R5 SELF-CHECKS PASSED")